# FEASE Training Quickstart

This notebook walks through an end-to-end training run of the FEASE (Feature-augmented EASE) recommender on a small synthetic dataset. You will:

1. Generate three long-format Parquet tables: `interactions`, `user_features`, `item_features`.
2. Train a FEASE model via the Rust backend (`rust_fease_recommender`).
3. Run warm-user, cold-start, and unknown-user predictions.
4. Explore More-Like-This (item similarity) and batch inference.
5. Save and reload the model artifact.

**Prerequisites.** Build and install the extension first:

```bash
.venv/bin/maturin develop
```

The Python package is `cr_fease` and the compiled module is `rust_fease_recommender`.

In [ ]:
import os
import time
import tempfile
from pathlib import Path

import numpy as np
import polars as pl

import rust_fease_recommender as fease

RNG = np.random.default_rng(seed=42)
WORK_DIR = Path(tempfile.mkdtemp(prefix="fease_quickstart_"))
print(f"Working directory: {WORK_DIR}")

## 1. Synthetic data

FEASE expects three long-format tables. We simulate a small streaming-media catalog:

- **Interactions** `(user_id, item_id, value)` — one row per user/item interaction, `value` is a positive weight (e.g. log watch-time).
- **User features** `(user_id, feature_name, value)` — side info like plan / device / region.
- **Item features** `(item_id, feature_name, value)` — side info like genre / type.

Users are drawn from a handful of latent personas; each persona prefers one or two genres. This gives the model real structure to learn.

In [ ]:
N_USERS = 200
N_ITEMS = 60
GENRES = ["Action", "Comedy", "Drama", "Romance", "SciFi"]
PLANS = ["Free", "Premium"]
DEVICES = ["Mobile", "Web", "Console", "TV"]
REGIONS = ["US", "EMEA", "APAC"]

# Persona -> (preferred_genre, second_genre)
PERSONAS = {
    "action_fan":   ("Action", "SciFi"),
    "comedy_fan":   ("Comedy", "Romance"),
    "drama_fan":    ("Drama", "Romance"),
    "scifi_nerd":   ("SciFi", "Action"),
}

# --- Items: each item gets a primary genre + a secondary tag ---
item_ids = [f"I{idx:03d}" for idx in range(N_ITEMS)]
item_primary_genre = {iid: GENRES[i % len(GENRES)] for i, iid in enumerate(item_ids)}
item_type = {iid: ("movie" if i % 2 == 0 else "episode") for i, iid in enumerate(item_ids)}

item_feat_rows = []
for iid in item_ids:
    item_feat_rows.append((iid, f"genre_{item_primary_genre[iid]}", 1.0))
    item_feat_rows.append((iid, f"type_{item_type[iid]}", 1.0))

df_items = pl.DataFrame(item_feat_rows, schema=["item_id", "feature_name", "value"], orient="row")

# --- Users: random persona + demographics ---
persona_names = list(PERSONAS.keys())
user_ids = [f"U{idx:04d}" for idx in range(N_USERS)]
user_persona = {uid: persona_names[RNG.integers(0, len(persona_names))] for uid in user_ids}
user_plan = {uid: PLANS[RNG.integers(0, len(PLANS))] for uid in user_ids}
user_device = {uid: DEVICES[RNG.integers(0, len(DEVICES))] for uid in user_ids}
user_region = {uid: REGIONS[RNG.integers(0, len(REGIONS))] for uid in user_ids}

user_feat_rows = []
for uid in user_ids:
    user_feat_rows.append((uid, f"plan_{user_plan[uid]}", 1.0))
    user_feat_rows.append((uid, f"device_{user_device[uid]}", 1.0))
    user_feat_rows.append((uid, f"region_{user_region[uid]}", 1.0))

df_user_features = pl.DataFrame(user_feat_rows, schema=["user_id", "feature_name", "value"], orient="row")

# --- Interactions: sample items biased toward the persona's preferred genres ---
inter_rows = []
for uid in user_ids:
    prefs = PERSONAS[user_persona[uid]]
    weights = np.array([
        3.0 if item_primary_genre[iid] == prefs[0]
        else 1.5 if item_primary_genre[iid] == prefs[1]
        else 0.3
        for iid in item_ids
    ])
    probs = weights / weights.sum()
    n_watched = int(RNG.integers(5, 20))
    picks = RNG.choice(item_ids, size=n_watched, replace=False, p=probs)
    for iid in picks:
        # log-watch-time-like positive weight
        inter_rows.append((uid, iid, float(RNG.uniform(1.0, 5.0))))

df_interactions = pl.DataFrame(inter_rows, schema=["user_id", "item_id", "value"], orient="row")

print(f"Interactions: {df_interactions.height} rows")
print(f"User features: {df_user_features.height} rows")
print(f"Item features: {df_items.height} rows")

In [ ]:
# Preview each table
print("--- Interactions ---")
print(df_interactions.head(5))
print("\n--- User features ---")
print(df_user_features.head(5))
print("\n--- Item features ---")
print(df_items.head(5))

In [ ]:
# Write to Parquet — the Rust training entrypoint reads from file paths
i_path = WORK_DIR / "interactions.parquet"
u_path = WORK_DIR / "user_features.parquet"
t_path = WORK_DIR / "item_features.parquet"

df_interactions.write_parquet(i_path)
df_user_features.write_parquet(u_path)
df_items.write_parquet(t_path)

print("Wrote:", i_path.name, u_path.name, t_path.name)

## 2. Train the model

The four core hyperparameters:

- `alpha` — weight on item-feature block (`T`). Increases how much item side-features shape the Gram matrix.
- `beta` — weight on user-feature block (`U`). Controls how much user side-features shape predictions.
- `lambda_` — L2 regularization. Typical production values are 100–150; smaller datasets need smaller values.
- `meta_weight` — diagonal weighting on metadata rows; `0.0` keeps metadata on equal footing with interactions.

In [ ]:
t0 = time.perf_counter()
model = fease.build_and_train(
    interactions_path=str(i_path),
    user_features_path=str(u_path),
    item_features_path=str(t_path),
    alpha=1.0,
    beta=1.0,
    lambda_=25.0,
    meta_weight=0.0,
)
train_s = time.perf_counter() - t0

print(f"Trained in {train_s:.2f}s")
print(f"  num_items           = {model.num_items}")
print(f"  num_user_features   = {model.num_user_features}")
print(f"  num_item_features   = {model.num_item_features}")

In [ ]:
# The Rust side also runs a structural validation pass. Surface it to Python.
passed, messages = model.validate()
print("validate() passed:", passed)
for m in messages:
    print(" -", m)

## 3. Predictions

Three archetypes to check:

1. **Warm user** — has past interactions and features.
2. **Cold-start user** — has no interactions, only features.
3. **Unknown user** — neither interactions nor features; scores should all be zero.

In [ ]:
# Warm user: pick one from our generated data
warm_uid = user_ids[0]
warm_inter = dict(
    df_interactions.filter(pl.col("user_id") == warm_uid)
    .select(["item_id", "value"]).iter_rows()
)
warm_feats = dict(
    df_user_features.filter(pl.col("user_id") == warm_uid)
    .select(["feature_name", "value"]).iter_rows()
)
print(f"User {warm_uid} persona={user_persona[warm_uid]} history_size={len(warm_inter)}")
print("Features:", warm_feats)

warm_recs = model.predict(warm_inter, warm_feats, top_k=10)
for guid, score in warm_recs:
    print(f"  {guid}  genre={item_primary_genre[guid]:<7s}  score={score:.4f}")

In [ ]:
# Cold-start user: only features, no interactions
cold_feats = {"plan_Premium": 1.0, "device_Mobile": 1.0, "region_US": 1.0}
cold_recs = model.predict({}, cold_feats, top_k=10)
print("Cold-start recs (plan=Premium, device=Mobile, region=US):")
for guid, score in cold_recs:
    print(f"  {guid}  genre={item_primary_genre[guid]:<7s}  score={score:.4f}")

In [ ]:
# Unknown user: no interactions, no recognizable features
unknown_recs = model.predict({}, {}, top_k=5)
print("Unknown user recs (should all be 0):")
for guid, score in unknown_recs:
    print(f"  {guid}  score={score:.4f}")

## 4. More-Like-This

`predict_similar_items` uses the item-item block of the S-matrix — the same learned weights used for user recommendations — so nearest neighbors are consistent with the recommendation surface.

In [ ]:
seed_item = item_ids[0]
similar = model.predict_similar_items(seed_item, top_k=5)
print(f"Items similar to {seed_item} (genre={item_primary_genre[seed_item]}):")
for guid, score in similar:
    print(f"  {guid}  genre={item_primary_genre[guid]:<7s}  score={score:.4f}")

## 5. Batch prediction

`predict_batch` takes a list of user dicts and avoids repeated Python/Rust boundary crossings — preferred for scoring many users at once.

In [ ]:
batch = [
    {"interactions": warm_inter, "features": warm_feats},
    {"interactions": {}, "features": {"plan_Free": 1.0, "device_TV": 1.0, "region_EMEA": 1.0}},
    {"interactions": {}, "features": {"plan_Premium": 1.0, "device_Console": 1.0, "region_APAC": 1.0}},
]

t0 = time.perf_counter()
batch_recs = model.predict_batch(batch, top_k=3)
batch_s = time.perf_counter() - t0
print(f"Batch of {len(batch)} users scored in {batch_s*1000:.2f}ms\n")

for i, recs in enumerate(batch_recs):
    print(f"User {i} top-3:", [(guid, round(score, 3)) for guid, score in recs])

## 6. Save and reload

The model serializes to a single binary file (magic bytes `FEAS`, versioned). Loading re-runs structural validation.

In [ ]:
artifact_path = WORK_DIR / "quickstart.fease"
model.save(str(artifact_path))
print(f"Saved to {artifact_path}  ({artifact_path.stat().st_size / 1024:.1f} KiB)")

reloaded = fease.load_model(str(artifact_path))
print(f"Reloaded: num_items={reloaded.num_items} user_feats={reloaded.num_user_features} item_feats={reloaded.num_item_features}")

# Predictions from the reloaded model should match the original bit-for-bit
orig = model.predict(warm_inter, warm_feats, top_k=5)
roundtrip = reloaded.predict(warm_inter, warm_feats, top_k=5)
for (g1, s1), (g2, s2) in zip(orig, roundtrip):
    assert g1 == g2 and abs(s1 - s2) < 1e-10
print("Round-trip predictions match.")

## Next steps

- **`02_evaluation_metrics.ipynb`** — compute Recall@K / NDCG@K / coverage with a held-out test set and run a hyperparameter sweep.
- **`03_deployment_artifact.ipynb`** — produce a production-grade model artifact with data-quality checks, manifest, and latency/qualitative spot-checks.